# Data Quality Analysis — Forearm Fracture Dataset

Checks:
1. **Duplicate detection** — exact (hash collision) and near-duplicates (hamming ≤ 5)
2. **Image quality flags** — dark, bright, low-contrast, wrong aspect ratio, tiny, corrupt
3. **Label consistency** — conflicting labels within a case, visually dissimilar series
4. **Split leakage** — case_id overlap between train/val/test, stratification check

**Output:** `logs/quality_report.json`

**Dependencies:** `pip install imagehash` (already in requirements.txt)

In [ ]:
import json
import sys
from pathlib import Path

import imagehash
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError
from tqdm.notebook import tqdm

# Resolve paths relative to notebook location
PROJECT_ROOT = Path("..").resolve()
SPLITS_CSV   = PROJECT_ROOT / "data" / "splits.csv"
LOG_DIR      = PROJECT_ROOT / "logs"
LOG_DIR.mkdir(exist_ok=True)

df = pd.read_csv(SPLITS_CSV)
print(f"Loaded {len(df):,} rows | columns: {df.columns.tolist()}")
print(f"Splits: {df['split'].value_counts().to_dict()}")
print(f"Cases : {df['case_id'].nunique():,}")

---
## 1. Duplicate Detection

In [ ]:
HASH_SIZE = 8  # 8×8 dhash = 64-bit fingerprint

def compute_dhash(path: str) -> str | None:
    """Difference hash — captures gradient structure, better than ahash for X-rays."""
    try:
        return str(imagehash.dhash(Image.open(path).convert("L"), hash_size=HASH_SIZE))
    except Exception:
        return None

print("Computing perceptual hashes for all images...")
df["dhash"] = df["path"].progress_apply(compute_dhash)

n_failed = df["dhash"].isna().sum()
print(f"\nHashing complete. Failed to open: {n_failed} images")

In [ ]:
# Exact duplicates — identical hash
hash_groups       = df.dropna(subset=["dhash"]).groupby("dhash")["path"].apply(list)
exact_dup_groups  = hash_groups[hash_groups.apply(len) > 1]
total_dup_images  = sum(len(g) - 1 for g in exact_dup_groups)

print(f"Exact duplicate groups : {len(exact_dup_groups)}")
print(f"Redundant images       : {total_dup_images}")

# Cross-split exact duplicates — potential leakage
cross_split_pairs = []
for h, paths in exact_dup_groups.items():
    splits = df[df["path"].isin(paths)]["split"].unique().tolist()
    if len(splits) > 1:
        cross_split_pairs.append({"hash": h, "splits": splits, "paths": paths})

print(f"\nCross-split exact duplicates (LEAKAGE RISK): {len(cross_split_pairs)}")
for item in cross_split_pairs:
    print(f"  hash={item['hash']}  splits={item['splits']}")

In [ ]:
# Near-duplicates within cases (Hamming distance ≤ 5)
# Full pairwise comparison (~11.5M pairs) is too slow — restrict to within-case pairs.
# 97% of cases have exactly 2 images, so within-case is O(2) per case.

HAMMING_THRESHOLD = 5

valid_df          = df.dropna(subset=["dhash"]).copy()
valid_df["hash_obj"] = valid_df["dhash"].apply(imagehash.hex_to_hash)

within_case_near_dupes = []

for case_id, group in tqdm(valid_df.groupby("case_id"), desc="Comparing series within cases"):
    rows = group.reset_index(drop=True)
    for i in range(len(rows)):
        for j in range(i + 1, len(rows)):
            dist = rows.loc[i, "hash_obj"] - rows.loc[j, "hash_obj"]
            if dist <= HAMMING_THRESHOLD:
                within_case_near_dupes.append({
                    "case_id": case_id,
                    "path_a":  rows.loc[i, "path"],
                    "path_b":  rows.loc[j, "path"],
                    "hamming": dist,
                    "split_a": rows.loc[i, "split"],
                    "split_b": rows.loc[j, "split"],
                })

nd_df = pd.DataFrame(within_case_near_dupes)
print(f"Within-case near-duplicate pairs (hamming ≤ {HAMMING_THRESHOLD}): {len(nd_df)}")
if len(nd_df):
    print(f"  Hamming distribution:\n{nd_df['hamming'].value_counts().sort_index().to_string()}")

In [ ]:
# Visualise 4 closest near-duplicate pairs
if len(nd_df) > 0:
    sample_pairs = nd_df.nsmallest(4, "hamming")
    fig, axes = plt.subplots(len(sample_pairs), 2, figsize=(10, 3 * len(sample_pairs)))
    if len(sample_pairs) == 1:
        axes = axes.reshape(1, -1)
    for row_idx, (_, row) in enumerate(sample_pairs.iterrows()):
        for col_idx, path in enumerate([row.path_a, row.path_b]):
            ax = axes[row_idx, col_idx]
            ax.imshow(Image.open(path).convert("L"), cmap="gray")
            ax.set_title(f"case={row.case_id}  hamming={row.hamming}\n{Path(path).name}", fontsize=7)
            ax.axis("off")
    fig.suptitle("Near-duplicate pairs (lowest Hamming distance)", fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print("No near-duplicates found within cases.")

---
## 2. Image Quality / Outlier Detection

In [ ]:
DARK_THRESHOLD       = 10    # mean pixel < 10  → nearly black
BRIGHT_THRESHOLD     = 245   # mean pixel > 245 → overexposed
LOW_CONTRAST_STD     = 15    # std < 15         → flat/uniform (artefact or blank)
MAX_ASPECT_RATIO     = 3.0   # w/h or h/w > 3  → corrupted geometry
MIN_DIMENSION        = 100   # any dimension < 100px → too small

quality_records = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Quality scan"):
    rec = dict(
        path=row["path"], case_id=row["case_id"], split=row["split"],
        label=row["label"], region=row["region"],
        flag_corrupt=False, flag_dark=False, flag_bright=False,
        flag_low_contrast=False, flag_aspect=False, flag_tiny=False,
        mean_px=None, std_px=None, width=None, height=None,
    )
    try:
        img  = Image.open(row["path"]).convert("L")
        arr  = np.array(img, dtype=np.float32)
        w, h = img.size
        mean_px = float(arr.mean())
        std_px  = float(arr.std())
        aspect  = max(w / h, h / w)
        rec.update(mean_px=round(mean_px, 2), std_px=round(std_px, 2), width=w, height=h)
        rec["flag_dark"]         = mean_px < DARK_THRESHOLD
        rec["flag_bright"]       = mean_px > BRIGHT_THRESHOLD
        rec["flag_low_contrast"] = std_px  < LOW_CONTRAST_STD
        rec["flag_aspect"]       = aspect  > MAX_ASPECT_RATIO
        rec["flag_tiny"]         = (w < MIN_DIMENSION or h < MIN_DIMENSION)
    except (UnidentifiedImageError, OSError):
        rec["flag_corrupt"] = True
    quality_records.append(rec)

qdf = pd.DataFrame(quality_records)
print("Scan complete.")

In [ ]:
flag_cols = [c for c in qdf.columns if c.startswith("flag_")]
flag_summary = qdf[flag_cols].sum().rename("count").to_frame()
flag_summary["pct"] = (flag_summary["count"] / len(qdf) * 100).round(2)
print("Quality flag summary:")
print(flag_summary.to_string())

print("\nFlags by split:")
print(qdf.groupby("split")[flag_cols].sum().to_string())

print("\nFlags by region + label:")
print(qdf.groupby(["region", "label"])[flag_cols].sum().to_string())

In [ ]:
# Pixel intensity and contrast distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(qdf["mean_px"].dropna(), bins=60, color="#4c9be8", edgecolor="white", lw=0.4)
axes[0].axvline(DARK_THRESHOLD,   color="red",    ls="--", label=f"Dark < {DARK_THRESHOLD}")
axes[0].axvline(BRIGHT_THRESHOLD, color="orange", ls="--", label=f"Bright > {BRIGHT_THRESHOLD}")
axes[0].set_xlabel("Mean pixel value (L channel)")
axes[0].set_ylabel("Images")
axes[0].set_title("Pixel brightness distribution")
axes[0].legend()

axes[1].hist(qdf["std_px"].dropna(), bins=60, color="#e05c5c", edgecolor="white", lw=0.4)
axes[1].axvline(LOW_CONTRAST_STD, color="red", ls="--", label=f"Low contrast < {LOW_CONTRAST_STD}")
axes[1].set_xlabel("Std of pixel values (contrast)")
axes[1].set_ylabel("Images")
axes[1].set_title("Pixel contrast distribution")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Show sample flagged images for visual review
for flag in ["flag_dark", "flag_bright", "flag_low_contrast", "flag_corrupt"]:
    flagged = qdf[qdf[flag] == True]
    if len(flagged) == 0:
        print(f"No images flagged as {flag}")
        continue
    sample = flagged.sample(min(4, len(flagged)), random_state=42)
    fig, axes = plt.subplots(1, len(sample), figsize=(4 * len(sample), 4))
    if len(sample) == 1:
        axes = [axes]
    for ax, (_, r) in zip(axes, sample.iterrows()):
        try:
            ax.imshow(Image.open(r["path"]).convert("L"), cmap="gray")
            ax.set_title(f"{flag}\nmean={r.mean_px}  std={r.std_px}", fontsize=7)
        except Exception:
            ax.set_title(f"CORRUPT\n{Path(r['path']).name}", fontsize=7)
        ax.axis("off")
    fig.suptitle(f"Sample: {flag} ({len(flagged)} total)", fontsize=11)
    plt.tight_layout()
    plt.show()

---
## 3. Label Consistency

In [ ]:
# Cases with conflicting labels across images (structurally impossible but worth auditing)
case_labels       = df.groupby("case_id")["label"].nunique()
multi_label_cases = case_labels[case_labels > 1]

print(f"Cases with conflicting labels: {len(multi_label_cases)}")
if len(multi_label_cases) > 0:
    print("ALERT — mixed labels within a case:")
    print(df[df["case_id"].isin(multi_label_cases.index)][
        ["case_id", "path", "label", "split"]
    ].to_string())
else:
    print("PASSED — all images within a case share the same label.")

In [ ]:
# Cases where both series look completely different (suspicious — may indicate mislabeling)
HIGH_DISSIMILARITY = 40  # hamming > 40/64 bits = very dissimilar

dissimilar_cases = []
for case_id, group in valid_df.groupby("case_id"):
    rows = group.reset_index(drop=True)
    if len(rows) < 2:
        continue
    dist = rows.loc[0, "hash_obj"] - rows.loc[1, "hash_obj"]
    if dist > HIGH_DISSIMILARITY:
        dissimilar_cases.append({
            "case_id": case_id, "hamming": dist,
            "label": rows.loc[0, "label"], "paths": rows["path"].tolist(),
        })

print(f"Cases with highly dissimilar series (hamming > {HIGH_DISSIMILARITY}): {len(dissimilar_cases)}")
if dissimilar_cases:
    print("\nTop 10 most dissimilar (review manually):")
    for c in sorted(dissimilar_cases, key=lambda x: -x["hamming"])[:10]:
        print(f"  case={c['case_id']}  hamming={c['hamming']}  label={c['label']}")

---
## 4. Split Leakage Verification

In [ ]:
# Case-level leakage check
case_split_map    = df.groupby("case_id")["split"].nunique()
multi_split_cases = case_split_map[case_split_map > 1]

print(f"Case_ids in more than one split: {len(multi_split_cases)}")
if len(multi_split_cases) > 0:
    print("CRITICAL — LEAKAGE DETECTED:")
    print(df[df["case_id"].isin(multi_split_cases.index)][["case_id", "split", "label"]].to_string())
else:
    print("PASSED — no case_id appears in multiple splits.")

print()
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    ids_a   = set(df[df["split"] == a]["case_id"])
    ids_b   = set(df[df["split"] == b]["case_id"])
    overlap = ids_a & ids_b
    status  = "FAIL" if overlap else "PASS"
    print(f"  {a} ∩ {b}: {len(overlap)} shared case_ids  [{status}]")

In [ ]:
# Stratification verification — fracture ratio per split should be within ±2%
global_ratio = (df["label"] == "fracture").mean()
TOLERANCE    = 0.02

print(f"Global fracture ratio: {global_ratio:.4f}")
print()
for split in ["train", "val", "test"]:
    sub      = df[df["split"] == split]
    ratio    = (sub["label"] == "fracture").mean()
    dev      = abs(ratio - global_ratio)
    status   = "PASS" if dev <= TOLERANCE else "FAIL"
    print(f"  {split:5s}: fracture_ratio={ratio:.4f}  deviation={dev:.4f}  [{status}]")

print("\nPer-region fracture ratio by split:")
region_split = (
    df.groupby(["split", "region"])
      .apply(lambda g: (g["label"] == "fracture").mean(), include_groups=False)
      .unstack("split")
      .round(3)
)
print(region_split.to_string())

---
## 5. Summary Report

In [ ]:
qdf["any_flag"] = qdf[flag_cols].any(axis=1)

report = {
    "total_images":             int(len(df)),
    "total_cases":              int(df["case_id"].nunique()),
    "exact_duplicate_groups":   int(len(exact_dup_groups)),
    "exact_redundant_images":   int(total_dup_images),
    "cross_split_exact_dups":   int(len(cross_split_pairs)),
    "within_case_near_dups":    int(len(nd_df)),
    "high_dissimilarity_cases": int(len(dissimilar_cases)),
    "quality_flags":            {col: int(qdf[col].sum()) for col in flag_cols},
    "total_flagged_images":     int(qdf["any_flag"].sum()),
    "leakage_detected":         bool(len(multi_split_cases) > 0),
    "label_conflicts":          int(len(multi_label_cases)),
    "stratification": {
        split: {
            "fracture_ratio": round(float((df[df["split"] == split]["label"] == "fracture").mean()), 4),
            "deviation":      round(float(abs((df[df["split"] == split]["label"] == "fracture").mean() - global_ratio)), 4),
        }
        for split in ["train", "val", "test"]
    },
    "generated_at": pd.Timestamp.now().isoformat(),
}

report_path = LOG_DIR / "quality_report.json"
report_path.write_text(json.dumps(report, indent=2))
print(f"Quality report saved → {report_path}")
print()
print(json.dumps({k: v for k, v in report.items() if k != "stratification"}, indent=2))

In [ ]:
# ── Optional: persist quality_flag column to splits.csv ──────────────────────
# Only run this cell if you want to write flags back to disk.
# The 'quality_flag' column is then used by FractureDataset to auto-exclude
# flagged images from the train split without re-running this notebook.

dup_paths = {p for paths in exact_dup_groups for p in paths}

def build_flag(row):
    qrow  = qdf[qdf["path"] == row["path"]]
    flags = []
    if len(qrow) and qrow.iloc[0][[c for c in flag_cols if c != "flag_corrupt"]].any():
        flags.append("outlier")
    if len(qrow) and qrow.iloc[0]["flag_corrupt"]:
        flags.append("corrupt")
    if row["path"] in dup_paths:
        flags.append("duplicate")
    return "|".join(flags)

df_out              = df.copy()
df_out["quality_flag"] = df_out.apply(build_flag, axis=1)
df_out.to_csv(SPLITS_CSV, index=False)

print(f"Wrote quality_flag to {SPLITS_CSV}")
print(df_out["quality_flag"].value_counts().to_string())